使用 IsolationForest 异常检测实现 Access-Log-Anomaly-Detection-Dataset 访问日志异常检测数据集 的分类检测

from sklearn.ensemble import IsolationForest


solationForest 是无监督异常检测算法，不是传统的有监督分类器。 我们可以用它把日志预测为“正常”或“异常”，再利用数据集的 target 标签评估效果。训练阶段不能把 target 当成模型特征。

另外，数据集已经有 anomaly_score 字段。为了避免模型直接利用可能由标签或规则生成的异常分数，我们先不使用 anomaly_score，也不使用 target 作为输入特征，仅从访问行为字段提取特征。这样更适合学习真实的无监督检测流程。


In [51]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

In [69]:
# 1. 加载用户分群 CSV 数据集
columns = ["timestamp","masked_user","source_ip","destination_ip","action","resource","protocol","access_result","location","device_type","login_attempt_count","anomaly_score","target","log_hour","day_of_week","month","time_diff_prev","user_activity_frequency","unique_actions","source_subnet","dest_subnet","resource_category","access_flag","protocol_HTTP","protocol_HTTPS","protocol_FTP","protocol_SSH","protocol_RDP","device_desktop","device_mobile","device_tablet","location_count","anomaly_bin","session_duration","event_count_session","session_complexity"]
# header=0表示第0行诗表头，这样 pandas 会把原文件第一行作为表头读取并跳过，
df = pd.read_csv("../../../data/ml/Access-Log-Anomaly-Detection-Dataset.csv", sep=",", header=0, names=columns)

# 删除第一行
print(df.shape)
df.head()


(5000, 36)


,timestamp,masked_user,source_ip,destination_ip,action,resource,protocol,access_result,location,device_type,...,protocol_SSH,protocol_RDP,device_desktop,device_mobile,device_tablet,location_count,anomaly_bin,session_duration,event_count_session,session_complexity
0,2024-10-07 12:36:46,user_3699,172.16.14.158,172.16.16.232,upload,/resource/6,SSH,failure,Sydney,mobile,...,1,0,0,1,0,34,low,924,12,8
1,2024-03-09 10:15:27,user_4957,172.16.25.23,172.16.105.151,login,/resource/87,HTTPS,success,London,desktop,...,0,0,1,0,0,50,low,1601,2,2
2,2024-09-12 13:48:10,user_2370,172.16.229.248,172.16.47.13,logout,/resource/23,HTTP,success,Tokyo,mobile,...,0,0,0,1,0,71,medium,1070,6,10
3,2024-12-21 11:42:17,user_4237,172.16.197.206,172.16.188.194,access,/resource/50,HTTP,success,Tokyo,desktop,...,0,0,1,0,0,67,low,2699,11,3
4,2024-12-05 17:58:26,user_5377,172.16.160.102,172.16.159.9,access,/resource/72,HTTP,success,Sydney,tablet,...,0,0,0,0,1,81,low,203,3,1


In [70]:
# 2. 检查字段、缺失值和 target 标签
print(df.isnull().sum())
print()
# 检查每一列的唯一值
print(df.nunique())

timestamp                  0
masked_user                0
source_ip                  0
destination_ip             0
action                     0
resource                   0
protocol                   0
access_result              0
location                   0
device_type                0
login_attempt_count        0
anomaly_score              0
target                     0
log_hour                   0
day_of_week                0
month                      0
time_diff_prev             0
user_activity_frequency    0
unique_actions             0
source_subnet              0
dest_subnet                0
resource_category          0
access_flag                0
protocol_HTTP              0
protocol_HTTPS             0
protocol_FTP               0
protocol_SSH               0
protocol_RDP               0
device_desktop             0
device_mobile              0
device_tablet              0
location_count             0
anomaly_bin                0
session_duration           0
event_count_se

In [71]:
# 3. 特征工程
#    ├── timestamp → hour、day_of_week、is_weekend
#    ├── source_ip → source_ip_prefix
#    ├── destination_ip → destination_ip_prefix
#    └── 保留 action、resource、device_type

# 查看每一列有多少个不同的值
print(df.nunique())
print()

df = df.drop(labels=["timestamp","masked_user","source_ip","destination_ip","resource"], axis=1)
print(df.nunique())
df

timestamp                  5000
masked_user                3821
source_ip                  4575
destination_ip             4561
action                        7
resource                    104
protocol                      4
access_result                 2
location                     10
device_type                   3
login_attempt_count          41
anomaly_score              3376
target                        2
log_hour                     14
day_of_week                   7
month                        12
time_diff_prev             2563
user_activity_frequency     100
unique_actions                4
source_subnet                 2
dest_subnet                   2
resource_category             4
access_flag                   2
protocol_HTTP                 2
protocol_HTTPS                2
protocol_FTP                  2
protocol_SSH                  2
protocol_RDP                  1
device_desktop                2
device_mobile                 2
device_tablet                 2
location

,action,protocol,access_result,location,device_type,login_attempt_count,anomaly_score,target,log_hour,day_of_week,...,protocol_SSH,protocol_RDP,device_desktop,device_mobile,device_tablet,location_count,anomaly_bin,session_duration,event_count_session,session_complexity
0,upload,SSH,failure,Sydney,mobile,8,0.0056,benign,12,Monday,...,1,0,0,1,0,34,low,924,12,8
1,login,HTTPS,success,London,desktop,10,0.1724,benign,10,Saturday,...,0,0,1,0,0,50,low,1601,2,2
2,logout,HTTP,success,Tokyo,mobile,7,0.4070,benign,13,Thursday,...,0,0,0,1,0,71,medium,1070,6,10
3,access,HTTP,success,Tokyo,desktop,3,0.1626,benign,11,Saturday,...,0,0,1,0,0,67,low,2699,11,3
4,access,HTTP,success,Sydney,tablet,6,0.0843,benign,17,Thursday,...,0,0,0,0,1,81,low,203,3,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,logout,SSH,failure,London,mobile,2,0.2747,benign,18,Thursday,...,1,0,0,1,0,29,low,996,2,7
4996,access,HTTP,success,London,tablet,7,0.3342,benign,14,Sunday,...,0,0,0,0,1,57,medium,439,15,10
4997,access,FTP,failure,Unknown,desktop,25,0.9215,malicious,4,Monday,...,0,0,1,0,0,6,high,25,29,10
4998,login,HTTP,failure,Berlin,desktop,1,0.4844,benign,16,Friday,...,0,0,1,0,0,85,medium,980,15,2


In [89]:
# 4. 拆分特征 X 和真实标签 y
#    ├── X：模型输入，不包含 target、anomaly_score
#    └── y：benign / anomaly_score，仅用于评估

y = df["target"]
X = df.drop(labels=["target", "anomaly_score"], axis=1)

print(X.shape, len(y))
# y_target = pd.DataFrame(y)
# y_anomaly_score = pd.DataFrame(y2)
print(X.shape, y.shape)

(5000, 29) 5000
(5000, 29) (5000,)


In [90]:
# 5. 划分训练集与测试集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=23, stratify=y)
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

(4000, 29) (1000, 29) (4000,) (1000,)


In [ ]:
# 6. 构建预处理器
#    ├── 数值特征：缺失值填充、标准化
#    └── 类别特征：缺失值填充、独热编码

numeric_columns = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_columns = X.select_dtypes(include=["str", "object", "category"]).columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler()),
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_columns),
    ("categorical", categorical_pipeline, categorical_columns),
])

model = Pipeline([
    ("preprocessor", preprocessor),
    ("detector", IsolationForest( n_estimators=200, contamination="auto", random_state=23, n_jobs=-1))
])

# 7. 训练 IsolationForest
#    └── 仅使用 X_train，不传入 y_train
# IsolationForest 是无监督异常检测算法。它主要根据样本在特征空间中的分布识别异常，不像逻辑回归那样通过 X_train 和 y_train 学习分类边界。
model.fit(X_train)



In [92]:
# 8. 对 X_test 预测
#    ├── predict() == 1  → 正常
#    └── predict() == -1 → 异常
y_pred_raw = model.predict(X_test)

In [101]:
# 9. 将正常/异常映射为 benign/malicious
# np.where(条件, 条件为 True 时的值, 条件为 False 时的值)
y_pred = np.where(y_pred_raw == 1, "benign", "malicious")
y_pred

array(['benign', 'benign', 'benign', 'benign', 'malicious', 'malicious',
       'malicious', 'malicious', 'malicious', 'malicious', 'malicious',
       'malicious', 'malicious', 'malicious', 'benign', 'malicious',
       'malicious', 'malicious', 'malicious', 'malicious', 'benign',
       'malicious', 'benign', 'benign', 'malicious', 'malicious',
       'malicious', 'malicious', 'benign', 'benign', 'malicious',
       'malicious', 'benign', 'malicious', 'malicious', 'benign',
       'malicious', 'malicious', 'malicious', 'malicious', 'malicious',
       'benign', 'malicious', 'benign', 'malicious', 'malicious',
       'malicious', 'benign', 'malicious', 'malicious', 'benign',
       'malicious', 'malicious', 'benign', 'malicious', 'malicious',
       'malicious', 'malicious', 'malicious', 'malicious', 'malicious',
       'malicious', 'malicious', 'malicious', 'malicious', 'malicious',
       'benign', 'benign', 'malicious', 'malicious', 'malicious',
       'malicious', 'malicious', 'ma

In [102]:
# 10. 与真实标签比较
#     ├── Accuracy
#     ├── Precision、Recall、F1
#     └── Confusion Matrix
y_test_arr = y_test.to_numpy()
print(classification_report(y_test_arr, y_pred))
print(confusion_matrix(y_test_arr, y_pred))
print(accuracy_score(y_test_arr, y_pred))


              precision    recall  f1-score   support

      benign       1.00      0.22      0.36       900
   malicious       0.12      1.00      0.22       100

    accuracy                           0.30      1000
   macro avg       0.56      0.61      0.29      1000
weighted avg       0.91      0.30      0.34      1000

[[196 704]
 [  0 100]]
0.296


In [113]:

# 1. 获取测试集对应的原始访问日志
results = df.loc[X_test.index].copy()

results["target"] = y_test
results["predict"] = y_pred
results.to_csv(
    "access_log_anomaly_results.csv",
    index=False,
    encoding="utf-8-sig"
)


In [118]:

# 上述训练模型准确度过低，这里使用不同的 contamination 进行训练试试

best_model = Pipeline([])
best_accuracy_score = 0
contaminations = [0.05, 0.1, 0.15, 0.2]
for contamination in contaminations:
    print(contamination)
    model = Pipeline([
        ("preprocessor", preprocessor),
        ("detector", IsolationForest( n_estimators=200, contamination=contamination, random_state=23, n_jobs=-1))
    ])

    model.fit(X_train)

    y_pred_raw = model.predict(X_test)
    y_pred = np.where(y_pred_raw == 1, "benign", "malicious")

    y_test_arr = y_test.to_numpy()
    accuracy = accuracy_score(y_test_arr, y_pred)
    print("accuracy_score: ", accuracy)
    if accuracy > best_accuracy_score:
        best_accuracy_score = accuracy
        best_model = model

    pass

y_pred_raw = best_model.predict(X_test)
y_pred = np.where(y_pred_raw == 1, "benign", "malicious")
y_test_arr = y_test.to_numpy()
accuracy = accuracy_score(y_test_arr, y_pred)
print("last accuracy_score: ", accuracy)
print(classification_report(y_test_arr, y_pred))
print(confusion_matrix(y_test_arr, y_pred))


0.05
accuracy_score:  0.958
0.1
accuracy_score:  0.995
0.15
accuracy_score:  0.946
0.2
accuracy_score:  0.884
last accuracy_score:  0.995
              precision    recall  f1-score   support

      benign       1.00      1.00      1.00       900
   malicious       0.97      0.98      0.98       100

    accuracy                           0.99      1000
   macro avg       0.98      0.99      0.99      1000
weighted avg       1.00      0.99      1.00      1000

[[897   3]
 [  2  98]]


In [122]:

"""
混淆矩阵解读


横向看（看一行）：真实标签相同的样本，被模型分别预测成了什么。
纵向看（看一列）：模型预测为同一类别的样本，真实标签分别是什么。

[[897   3]
 [  2  98]]


混淆矩阵
列：模型预测结果
行 = 真实标签；列 = 预测标签。绿色表示预测正确，橙色表示预测错误。

                 预测 benign        预测 malicious      合计
真实 benign       897(正常 → 正常)    3(正常 → 恶意)      900
真实 malicious    2(恶意 → 正常)      98(恶意 → 恶意)     100
合计              899                101               1000

行解读：
    真实benign有900个，预测为benign的有897个，预测为malicious有3个，正确率=897/900=99.5%
    真实malicious有100个，预测为有2个，预测为malicious的有98个，正确率=98/100=98%
整体解读：
    整体正确率=(897+98)/1000=0.995
    恶意访问 Recall: 98%
    恶意访问 Precision: 97.03%
    正常访问误报率: 0.33%



机器学习中常用 TP、TN、FP、FN 表示混淆矩阵中的四种情况。通常把 malicious 设为正类：
符号	数量	含义
TP	98	恶意访问，正确预测为恶意
TN	897	正常访问，正确预测为正常
FP	3	正常访问，被误报为恶意
FN	2	恶意访问，被漏报为正常
可以这样记：
- TP：该抓的抓到了。
- TN：该放过的正确放过了。
- FP：误报，把正常访问当成恶意。
- FN：漏报，把恶意访问当成正常。
因此：
- Accuracy：\((TP+TN)/1000=(98+897)/1000=99.5\%\)
- Precision：\(TP/(TP+FP)=98/(98+3)\approx97.03\%\)
- Recall：\(TP/(TP+FN)=98/(98+2)=98\%\)
"""


0.9966666666666667
0.995
0.0033333333333333335
